# Train on final LightGBM params (Phase 5, step 1)

Trains the **final model** with the settings chosen in tuning step 2 (`08_lgbm_tuning_step2.ipynb`), then dry-runs it
on month 5 at the rules' weekly budget.

| § | What | Time |
|---|---|---|
| 0 | Settings (**the toggles are here**) and setup | seconds |
| 1 | The tuned settings | seconds |
| 2 | Train the final model on months 0–4 | a few minutes, skipped when `RETRAIN = False` |
| 3 | Dry run on month 5 at the rules' budget, against the default-settings dry run | about a minute |

**The model** is the one frozen on 28 Oct 2026: LightGBM on the 38 inputs (28 store features, 8 `roll4` features,
`weeks_since_open`, `app_score`), fitted on **all** accounts opened in months 0–4, weeks 0–26, closed weeks dropped.
Training is done by the team's `models.train.train` (the same code as `python -m models train`), so this notebook only
feeds it the tuned settings. The model is saved to `data/<run>/models/<MODEL_NAME>/` (git-ignored); its manifest and
feature importances go to `outputs/modelling/models/<run>/<MODEL_NAME>/` (committed).

**Rules kept.** Training reads months 0–4 only. Month 5 is read by the dry run, which is scoring code
(`evaluation.score`); months 6–7 are not touched (they are scored once, on 5 Nov 2026, with `--final-test`). The
settings were chosen on the folds of months 0–4, so this dry run checks them; it must not be used to pick between
settings. Calibration on month 5 (due 4 Nov 2026) is a separate step, not done here.

## 0. Settings and setup

In [ ]:
import os

RETRAIN = True          # True: train the model (a few minutes). False: use the model already saved under MODEL_NAME
RUN_DRYRUN = True       # True: score month 5 at the rules' budget with the saved model (about a minute)
MODEL_NAME = "frozen_tuned"                          # model folder; "frozen" is the default-settings model
DATA = os.environ.get("EDA_DATA", "data/v1_full")

In [ ]:
import json, sys, time
from pathlib import Path

import pandas as pd
import matplotlib
if "ipykernel" not in sys.modules:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt

# find the repo root (the folder holding src/amldata) so Run All works from notebooks/
ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "amldata").is_dir():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("start this notebook from inside the aml-detection-system repo")
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import lightgbm as lgb
import models.train as train
import evaluation.score as score

RUN = os.path.basename(os.path.normpath(DATA))
TUNING = Path(f"outputs/modelling/models/{RUN}/lgbm_tuning_step2")
REPORT = Path(f"outputs/modelling/models/{RUN}/{MODEL_NAME}")
RESULTS = Path(f"outputs/modelling/results/{RUN}")
DRYRUN_NAME = f"month5_dryrun_{MODEL_NAME}"
show = display if "ipykernel" in sys.modules else print
print(f"data {DATA} | model {MODEL_NAME} | RETRAIN = {RETRAIN}, RUN_DRYRUN = {RUN_DRYRUN} | lightgbm {lgb.__version__}")

## 1. The tuned settings

Read from `tuned_params.json` (written by notebook 08). If only `best_params.json` exists, the settings are taken from
it and `tuned_params.json` is written. They are merged over the fixed settings of `models.train`
(`class_weight="balanced"`, `random_state=42`, `deterministic=True`, `force_row_wise=True`).

In [ ]:
tuned_file = TUNING / "tuned_params.json"
if tuned_file.exists():
    PARAMS = json.loads(tuned_file.read_text())
else:
    best = TUNING / "best_params.json"
    if not best.exists():
        raise FileNotFoundError(f"neither {tuned_file} nor {best} found: run notebook 08 first")
    PARAMS = {**json.loads(best.read_text())["params"], "subsample_freq": 1}
    tuned_file.write_text(json.dumps(PARAMS, indent=2))
    print(f"wrote {tuned_file}")
for k in ("n_estimators", "num_leaves", "min_child_samples", "subsample_freq"):
    PARAMS[k] = int(PARAMS[k])
show(pd.Series({**train.DEFAULT_PARAMS, **PARAMS}, name="setting", dtype=object).to_frame())

## 2. Train the final model on months 0–4

One fit on every training account (about 4.4M account-weeks on `v1_full`); the application score is out of fold for
the training accounts and fitted on all of them for scoring new accounts, as in `models.train`. With `RETRAIN = False`
the saved model is loaded instead, after `models.train.load` checks it still matches the data, the store and the
frozen inputs.

In [ ]:
if RETRAIN:
    t = time.time()
    man = train.train(DATA, MODEL_NAME, PARAMS)
    print(f"trained {MODEL_NAME}: {man['rows']:,} account-weeks ({man['mule_rows']:,} mule weeks), "
          f"{man['accounts']:,} accounts, {man['n_inputs']} inputs, {(time.time() - t) / 60:.1f} min")
else:
    man = train.load(DATA, MODEL_NAME)[2]
    print(f"loaded {MODEL_NAME} (trained on {man['rows']:,} account-weeks, settings from {man['params_source']})")
if man["params_source"] != "file" or man["lightgbm_params"].get("learning_rate") != PARAMS["learning_rate"]:
    print("WARNING: this model was not trained with the tuned settings: set RETRAIN = True")

imp = pd.read_csv(REPORT / "importance.csv").head(15).iloc[::-1]
fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(imp["feature"], imp["share"], color="#1baf7a")
for y, v in enumerate(imp["share"]):
    ax.text(v, y, f" {v:.1%}", va="center", fontsize=8)
ax.set(xlabel="share of total gain", title=f"Top 15 inputs, {MODEL_NAME}")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(REPORT / "01_importance.png", dpi=130)
plt.show() if "ipykernel" in sys.modules else plt.close(fig)

## 3. Dry run on month 5 at the rules' budget

`evaluation.score` (the Phase 5 harness): every account opened in month 5, a clock of weeks since opening, and each
week the model flags as many accounts as the rules alerted on that cohort that week; flags are permanent. Horizon
weeks 0–26 (main) and 0–77 (appendix). Results go to `outputs/modelling/results/<run>/month5_dryrun_<MODEL_NAME>/`.

The table puts the tuned model beside the team's default-settings dry run (`month5_dryrun`, 28 Oct 2026) and the
rules. Month 5 has 97 launderers (78 mules by week 26), so one launderer is about 1.3 points of recall: small
differences are noise.

In [ ]:
COLS = {"flagged": "flags", "caught": "caught", "mules": "mules", "recall": "recall", "precision": "precision",
        "median_days_from_start": "median days from start", "catch_mean": "catch_mean",
        "caught_fake": "fake", "caught_witting": "witting", "caught_unwitting": "unwitting"}


def summary(folder: Path, w: int) -> pd.DataFrame:
    return pd.read_csv(folder / f"summary_w{w}.csv").set_index("policy")


if RUN_DRYRUN:
    t = time.time()
    score.run(DATA, [5], [26, 77], DRYRUN_NAME, model_name=MODEL_NAME)
    print(f"dry run done in {time.time() - t:.0f}s -> {RESULTS / DRYRUN_NAME}")

tuned_dir, default_dir = RESULTS / DRYRUN_NAME, RESULTS / "month5_dryrun"
if tuned_dir.exists():
    for w in (26, 77):
        t_ = summary(tuned_dir, w)
        rows = {f"model, tuned ({MODEL_NAME})": t_.loc["model"]}
        if default_dir.exists():
            rows["model, default settings (frozen)"] = summary(default_dir, w).loc["model"]
        rows["rules"] = t_.loc["rules"]
        table = pd.DataFrame(rows).T[list(COLS)].rename(columns=COLS)
        print(f"\nweeks 0-{w}" + (" (main horizon)" if w == 26 else " (appendix)"))
        show(table.round(3))
    h2h = json.loads((tuned_dir / "settings.json").read_text())["head_to_head"]["26"]
    print(f"\nweeks 0-26, head to head: {h2h['both']} caught by both, {h2h['model_only']} by the model only, "
          f"{h2h['rules_only']} by the rules only, {h2h['neither']} by neither; when both catch a launderer the "
          f"model is first {h2h['model_first_when_both']} times, the rules {h2h['rules_first_when_both']}")
    if not default_dir.exists():
        print(f"(no default-settings dry run in {default_dir} to compare with)")
else:
    print("no dry run yet: set RUN_DRYRUN = True")